In [76]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
df = pd.read_csv("data.csv", encoding="ISO-8859-1")
print("Dataset loaded successfully!")
print("Original shape:", df.shape)

Dataset loaded successfully!
Original shape: (541909, 8)


In [77]:
df = df.drop_duplicates()
df = df.dropna(subset=["Description"])
print("After cleaning:", df.shape)
print("Missing values:")
print(df.isnull().sum())

After cleaning: (535187, 8)
Missing values:
InvoiceNo           0
StockCode           0
Description         0
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     133583
Country             0
dtype: int64


In [78]:
df["InvoiceDate"] = pd.to_datetime(
    df["InvoiceDate"]
)
df["TotalAmount"] = (
    df["Quantity"] * df["UnitPrice"]
)
print(df[[
    "InvoiceNo",
    "Description",
    "Quantity",
    "UnitPrice",
    "TotalAmount"
]].head())

  InvoiceNo                          Description  Quantity  UnitPrice  \
0    536365   WHITE HANGING HEART T-LIGHT HOLDER         6       2.55   
1    536365                  WHITE METAL LANTERN         6       3.39   
2    536365       CREAM CUPID HEARTS COAT HANGER         8       2.75   
3    536365  KNITTED UNION FLAG HOT WATER BOTTLE         6       3.39   
4    536365       RED WOOLLY HOTTIE WHITE HEART.         6       3.39   

   TotalAmount  
0        15.30  
1        20.34  
2        22.00  
3        20.34  
4        20.34  


In [79]:
df = df[
    (~df["InvoiceNo"].astype(str).str.startswith("C")) &
    (df["Quantity"] > 0) &
    (df["UnitPrice"] > 0)
]
print("Final cleaned shape:", df.shape)

Final cleaned shape: (524878, 9)


In [80]:
print("Total Transactions:", df["InvoiceNo"].nunique())
print("Total Products:", df["StockCode"].nunique())
print("Total Customers:", df["CustomerID"].nunique())
print("Total Countries:", df["Country"].nunique())
print("Total Revenue:", round(df["TotalAmount"].sum(), 2))

Total Transactions: 19960
Total Products: 3922
Total Customers: 4338
Total Countries: 38
Total Revenue: 10642110.8


In [81]:
total_revenue = df["TotalAmount"].sum()
total_orders = df["InvoiceNo"].nunique()
total_customers = df["CustomerID"].nunique()
total_products = df["StockCode"].nunique()
average_order_value = total_revenue / total_orders

print("Total Revenue:", round(total_revenue, 2))
print("Total Orders:", total_orders)
print("Total Customers:", total_customers)
print("Total Products:", total_products)
print("Average Order Value:", round(average_order_value, 2))

Total Revenue: 10642110.8
Total Orders: 19960
Total Customers: 4338
Total Products: 3922
Average Order Value: 533.17


In [82]:
df["Month"] = df["InvoiceDate"].dt.to_period("M").astype(str)

monthly_sales = df.groupby("Month")["TotalAmount"].sum().reset_index()

monthly_sales.columns = ["Month", "Revenue"]

display(monthly_sales)

,Month,Revenue
0,2010-12,821452.730
1,2011-01,689811.610
2,2011-02,522545.560
3,2011-03,716215.260
4,2011-04,536968.491
5,2011-05,769296.610
6,2011-06,760547.010
7,2011-07,718076.121
8,2011-08,757841.380
9,2011-09,1056435.192


In [83]:
top_products = (
    df.groupby("Description")["Quantity"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

display(top_products)

,Description,Quantity
0,"PAPER CRAFT , LITTLE BIRDIE",80995
1,MEDIUM CERAMIC TOP STORAGE JAR,78033
2,WORLD WAR 2 GLIDERS ASSTD DESIGNS,54951
3,JUMBO BAG RED RETROSPOT,48371
4,WHITE HANGING HEART T-LIGHT HOLDER,37872
5,POPCORN HOLDER,36749
6,PACK OF 72 RETROSPOT CAKE CASES,36396
7,ASSORTED COLOUR BIRD ORNAMENT,36362
8,RABBIT NIGHT LIGHT,30739
9,MINI PAINT SET VINTAGE,26633


In [84]:
country_sales = (
    df.groupby("Country")["TotalAmount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

display(country_sales.head(10))

,Country,TotalAmount
0,United Kingdom,9001744.094
1,Netherlands,285446.340
2,EIRE,283140.520
3,Germany,228678.400
4,France,209625.370
5,Australia,138453.810
6,Spain,61558.560
7,Switzerland,57067.600
8,Belgium,41196.340
9,Sweden,38367.830


In [85]:
fig1 = px.line(
    monthly_sales,
    x="Month",
    y="Revenue",
    title="Monthly Revenue Trend",
    markers=True
)

fig1.update_layout(
    template="plotly_white",
    xaxis_title="Month",
    yaxis_title="Revenue (£)"
)

fig1.show()

In [86]:
fig2 = px.bar(
    top_products,
    x="Description",
    y="Quantity",
    title="Top 10 Best-Selling Products",
    color="Quantity",
    color_continuous_scale="Blues"
)

fig2.update_layout(
    template="plotly_white",
    xaxis_title="Product",
    yaxis_title="Quantity Sold",
    xaxis_tickangle=-45
)

fig2.show()

In [87]:
top_countries = country_sales.head(10)

fig3 = px.pie(
    top_countries,
    names="Country",
    values="TotalAmount",
    title="Top 10 Countries by Revenue",
    hole=0.35
)

fig3.update_layout(
    template="plotly_white"
)

fig3.show()

In [88]:
fig4 = px.histogram(
    df[df["UnitPrice"] <= 50],
    x="UnitPrice",
    nbins=60,
    title="Product Price Distribution (Unit Price ≤ £50)",
    color_discrete_sequence=["royalblue"]
)

fig4.update_layout(
    template="plotly_white",
    xaxis_title="Unit Price (£)",
    yaxis_title="Number of Records"
)

fig4.show()

In [89]:
correlation = df[
    ["Quantity", "UnitPrice", "TotalAmount"]
].corr()

fig5 = px.imshow(
    correlation,
    text_auto=".2f",
    color_continuous_scale="blues",
    title="E-Commerce Sales Correlation Heatmap",
    zmin=-1,
    zmax=1
)

fig5.update_layout(
    template="plotly_white"
)

fig5.show()

In [90]:
import plotly.graph_objects as go

# Monthly revenue for all countries combined
all_monthly = (
    df.groupby("Month")["TotalAmount"]
    .sum()
    .reset_index()
)

# Monthly revenue for individual countries
monthly_country = (
    df.groupby(["Country", "Month"])["TotalAmount"]
    .sum()
    .reset_index()
)

countries = sorted(monthly_country["Country"].unique())

fig6 = go.Figure()

# First trace: All countries combined
fig6.add_trace(
    go.Scatter(
        x=all_monthly["Month"],
        y=all_monthly["TotalAmount"],
        mode="lines+markers",
        name="All Countries",
        line=dict(color="blue", width=3),
        visible=True
    )
)

# Add individual country traces
for country in countries:

    country_data = monthly_country[
        monthly_country["Country"] == country
    ]

    fig6.add_trace(
        go.Scatter(
            x=country_data["Month"],
            y=country_data["TotalAmount"],
            mode="lines+markers",
            name=country,
            visible=False
        )
    )

# Dropdown buttons
buttons = []

buttons.append(
    dict(
        label="All Countries",
        method="update",
        args=[
            {"visible": [True] + [False] * len(countries)},
            {"title": "Monthly Revenue - All Countries"}
        ]
    )
)

for i, country in enumerate(countries):

    visibility = [False] * (len(countries) + 1)
    visibility[i + 1] = True

    buttons.append(
        dict(
            label=country,
            method="update",
            args=[
                {"visible": visibility},
                {"title": f"Monthly Revenue - {country}"}
            ]
        )
    )

# Dashboard layout
fig6.update_layout(
    title="Monthly Revenue - All Countries",
    template="plotly_white",
    xaxis_title="Month",
    yaxis_title="Revenue (£)",
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=1,
            y=1.15,
            xanchor="right",
            yanchor="top"
        )
    ],
    height=550
)

fig6.show()

In [91]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px
import pandas as pd
import numpy as np

print("Dashboard libraries ready!")

Dashboard libraries ready!


In [92]:
total_revenue = df["TotalAmount"].sum()
total_orders = df["InvoiceNo"].nunique()
total_customers = df["CustomerID"].nunique()
total_products = df["StockCode"].nunique()

average_order_value = total_revenue / total_orders

print("Revenue:", round(total_revenue, 2))
print("Orders:", total_orders)
print("Customers:", total_customers)
print("Products:", total_products)
print("Average Order Value:", round(average_order_value, 2))

Revenue: 10642110.8
Orders: 19960
Customers: 4338
Products: 3922
Average Order Value: 533.17


In [93]:

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ==========================================
# 1. PREPARE DATA
# ==========================================

# Prepare month column
if "Month" not in df.columns:
    df["Month"] = pd.to_datetime(
        df["InvoiceDate"]
    ).dt.to_period("M").astype(str)

month_dates = pd.to_datetime(
    df["Month"].astype(str).str[:7] + "-01"
)

data = df.assign(_MonthDate=month_dates)

# Monthly revenue
monthly_sales = (
    data.groupby("_MonthDate")["TotalAmount"]
    .sum()
    .reset_index()
    .sort_values("_MonthDate")
)

# Country-wise monthly revenue
monthly_country = (
    data.groupby(["Country", "_MonthDate"])["TotalAmount"]
    .sum()
    .reset_index()
)

# Top 10 products
top_products = (
    df.groupby("Description")["Quantity"]
    .sum()
    .nlargest(10)
    .sort_values()
    .reset_index()
)

top_products["ShortName"] = (
    top_products["Description"].str.slice(0, 17) + "..."
)

# Top 10 countries
country_sales = (
    df.groupby("Country")["TotalAmount"]
    .sum()
    .nlargest(10)
    .reset_index()
)

# Correlation
correlation = df[
    ["Quantity", "UnitPrice", "TotalAmount"]
].corr()

# KPI calculations
total_revenue = df["TotalAmount"].sum()
total_orders = df["InvoiceNo"].nunique()
total_customers = df["CustomerID"].nunique()
total_products = df["StockCode"].nunique()

# ==========================================
# 2. CREATE DASHBOARD LAYOUT
# ==========================================

fig = make_subplots(
    rows=3,
    cols=2,

    specs=[
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "domain"}, {"type": "xy"}],
        [{"type": "xy", "colspan": 2}, None]
    ],

    column_widths=[0.45, 0.55],

    row_heights=[0.34, 0.32, 0.34],

    horizontal_spacing=0.16,
    vertical_spacing=0.16,

    subplot_titles=[
        "Monthly Revenue Trend",
        "Top 10 Best-Selling Products",
        "Revenue by Country",
        "Product Price Distribution",
        "Sales Correlation Heatmap"
    ]
)

# ==========================================
# 3. LINE CHART
# ==========================================

fig.add_trace(
    go.Scatter(
        x=monthly_sales["_MonthDate"],
        y=monthly_sales["TotalAmount"],
        mode="lines+markers",
        name="All Countries",
        line=dict(color="#2563EB", width=3),
        marker=dict(size=7),
        hovertemplate=
        "Month: %{x|%b %Y}<br>"
        "Revenue: £%{y:,.2f}<extra></extra>"
    ),
    row=1, col=1
)

# ==========================================
# 4. BAR CHART
# ==========================================

fig.add_trace(
    go.Bar(
        x=top_products["Quantity"],
        y=top_products["ShortName"],
        orientation="h",
        customdata=top_products["Description"],
        marker_color="#3B82F6",
        hovertemplate=
        "Product: %{customdata}<br>"
        "Quantity: %{x:,}<extra></extra>"
    ),
    row=1, col=2
)

# ==========================================
# 5. PIE CHART
# ==========================================

fig.add_trace(
    go.Pie(
        labels=country_sales["Country"],
        values=country_sales["TotalAmount"],
        hole=0.45,
        textinfo="percent",
        textposition="inside",
        sort=True,
        hovertemplate=
        "Country: %{label}<br>"
        "Revenue: £%{value:,.2f}<extra></extra>"
    ),
    row=2, col=1
)

# ==========================================
# 6. HISTOGRAM
# ==========================================

fig.add_trace(
    go.Histogram(
        x=df.loc[
            df["UnitPrice"].between(0, 50),
            "UnitPrice"
        ],
        nbinsx=35,
        marker_color="#60A5FA",
        name="Price Distribution"
    ),
    row=2, col=2
)

# ==========================================
# 7. HEATMAP
# ==========================================

fig.add_trace(
    go.Heatmap(
        z=correlation.values,
        x=correlation.columns,
        y=correlation.index,
        text=np.round(correlation.values, 2),
        texttemplate="%{text}",
        colorscale="Blues",
        zmin=-1,
        zmax=1,
        showscale=False
    ),
    row=3, col=1
)

# ==========================================
# 8. ADD COUNTRY FILTER
# ==========================================

countries = sorted(df["Country"].dropna().unique())

base_count = len(fig.data)

# Add country-specific revenue lines
for country in countries:

    country_data = monthly_country[
        monthly_country["Country"] == country
    ].sort_values("_MonthDate")

    fig.add_trace(
        go.Scatter(
            x=country_data["_MonthDate"],
            y=country_data["TotalAmount"],
            mode="lines+markers",
            name=country,
            visible=False,
            line=dict(width=3),
            hovertemplate=
            "Country: " + country +
            "<br>Month: %{x|%b %Y}" +
            "<br>Revenue: £%{y:,.2f}<extra></extra>"
        ),
        row=1, col=1
    )

# Dropdown buttons
buttons = []

# All countries
visibility = [True] * base_count + [False] * len(countries)

buttons.append(
    dict(
        label="All Countries",
        method="update",
        args=[{"visible": visibility}]
    )
)

# Individual countries
for i, country in enumerate(countries):

    visibility = [True] * base_count + [False] * len(countries)

    visibility[0] = False
    visibility[base_count + i] = True

    buttons.append(
        dict(
            label=country,
            method="update",
            args=[{"visible": visibility}]
        )
    )

# ==========================================
# 9. AXIS FORMATTING
# ==========================================

fig.update_xaxes(
    title_text="Month",
    tickformat="%b %Y",
    row=1, col=1
)

fig.update_yaxes(
    title_text="Revenue (£)",
    row=1, col=1
)

fig.update_xaxes(
    title_text="Quantity Sold",
    row=1, col=2
)

fig.update_yaxes(
    title_text="",
    tickfont=dict(size=9),
    row=1, col=2
)

fig.update_xaxes(
    title_text="Unit Price (£)",
    row=2, col=2
)

fig.update_yaxes(
    title_text="Number of Records",
    row=2, col=2
)

# ==========================================
# 10. PROFESSIONAL DASHBOARD DESIGN
# ==========================================

fig.update_layout(
    template="plotly_white",

    width=1400,
    height=1600,

    font=dict(
        family="Arial",
        size=12,
        color="#334155"
    ),

    showlegend=False,

    paper_bgcolor="#FFFFFF",
    plot_bgcolor="#FFFFFF",

    margin=dict(
        top=300,
        bottom=100,
        left=90,
        right=90
    ),

    title=dict(
        text="E-COMMERCE SALES ANALYTICS DASHBOARD",
        x=0.5,
        y=0.98,
        xanchor="center",
        yanchor="top",
        font=dict(
            size=27,
            color="#1E3A8A"
        )
    ),

    # Country dropdown in its own top area
    updatemenus=[
        dict(
            buttons=buttons,
            direction="down",
            x=0.5,
            y=1.14,
            xanchor="center",
            yanchor="top",
            showactive=True,
            bgcolor="#EFF6FF",
            bordercolor="#93C5FD",
            font=dict(size=12)
        )
    ]
)

# ==========================================
# 11. KPI CARDS
# ==========================================

kpis = [
    ("TOTAL REVENUE", f"£{total_revenue:,.2f}"),
    ("TOTAL ORDERS", f"{total_orders:,}"),
    ("TOTAL CUSTOMERS", f"{total_customers:,}"),
    ("TOTAL PRODUCTS", f"{total_products:,}")
]

positions = [0.12, 0.37, 0.63, 0.88]

for (label, value), x in zip(kpis, positions):

    fig.add_annotation(
        x=x,
        y=1.22,
        xref="paper",
        yref="paper",

        text=(
            f"<b>{label}</b><br>"
            f"<span style='font-size:22px'>{value}</span>"
        ),

        showarrow=False,
        align="center",

        font=dict(
            size=14,
            color="#1E40AF"
        ),

        bgcolor="#EFF6FF",
        bordercolor="#BFDBFE",
        borderwidth=1,
        borderpad=13
    )

# ==========================================
# 12. DISPLAY FINAL DASHBOARD
# ==========================================

fig.show()


In [94]:
# Save the complete dashboard as index.html
fig.write_html(
    "index.html",
    include_plotlyjs=True
)

# Download the HTML file
from google.colab import files
files.download("index.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>